# Detector acceptance curves

This notebook plots the benchmark acceptance curves for DEAP-3600, DarkSide-50, DarkSide-20k, LZ, and XENONnT in independent figures.

The curves are approximate digitizations stored in `src.detectors.reference_detectors`. The DEAP curve is plotted in nuclear-recoil energy (`keVnr`), using an ER light yield of 6.1 PE/keVee and a constant argon quenching factor Q = E_ee/E_nr = 0.25.

In [ ]:
%load_ext autoreload
%autoreload 2


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import matplotlib.pyplot as plt
from src.plotting.plot_detectors import plot_detector_acceptance

from src.detectors.reference_detectors import (
    _deap_acceptance,
    _ds20k_acceptance,
    _ds50_acceptance,
    _lz_acceptance,
    _xenont_acceptance,
)

## Prepare the curves

The curves are plotted using the energy variable provided by `reference_detectors`. For DEAP, the approximate conversion is E_nr [keVnr] = PE / (6.1 * 0.25). The constant quenching factor is a benchmark approximation, not the energy-dependent NR calibration used in the published analysis. LZ is shown in two separate figures: the extended acceptance with the HE sideband (`include_he_sb=True`) and the restricted WS acceptance without it (`include_he_sb=False`).

In [ ]:
acceptance_functions = {
    "DEAP-3600": _deap_acceptance,
    "DarkSide-50": _ds50_acceptance,
    "DarkSide-20k": _ds20k_acceptance,
    "LZ (extended, HE SB)": lambda: _lz_acceptance(include_he_sb=True),
    "LZ (restricted, WS only)": lambda: _lz_acceptance(include_he_sb=False),
    "XENONnT": _xenont_acceptance,
}

curves = {
    detector: acceptance_function()
    for detector, acceptance_function in acceptance_functions.items()
}

In [ ]:
x_limits = {
    "DEAP-3600": 1500,
    "DarkSide-50": 1200,
    "DarkSide-20k": 1050,
    "LZ (extended, HE SB)": 750,
    "LZ (restricted, WS only)": 320,
    "XENONnT": 1050,
}

for detector, acceptance_curve in curves.items():
    fig, ax = plt.subplots(figsize=(9, 6))
    fig, ax = plot_detector_acceptance(
        acceptance_curve,
        ax=ax,
        color="black",
        marker=None if detector.startswith("LZ") else "o",
        label=detector,
    )

    ax.axvspan(30, 80, color="gray", alpha=0.08, label="30–80 keV reference band")
    ax.set(
        xlabel=(
            "NR recoil energy [keVnr] (approx. PE mapping, Q=0.25)"
            if detector == "DEAP-3600"
            else "Recoil energy [keV]"
        ),
        title=f"{detector} acceptance benchmark",
        xlim=(0, x_limits[detector]),
    )
    ax.legend()
    fig.tight_layout()
    plt.show()